In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('NO HAY GPU. Ve a Entorno de ejecución > Cambiar tipo > GPU T4')


PyTorch: 2.10.0+cu128
CUDA disponible: True
GPU: Tesla T4
VRAM: 15.6 GB


Esta celda verifica si se está utilizando la GPU, pues el modelo es grande y de otra manera no hay suficiente espacio.

In [ ]:
!pip install bitsandbytes accelerate transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


Esta celda instala las dependencias y librerías necesarias para el notebook
- bitsandbytes carga modelos más grandes usando menos memoria por medio de la cuantización, con representaciones más compactas de 4 u 8bits
- accelerate maneja dispositivos de manera más eficiente
- transformers es una librería de Hugging Face para poder cargar los modelos

In [ ]:
from google.colab import drive, files
import torch
import numpy as np
import os
import gc

drive.mount("/content/drive")
save_path = "/content/drive/MyDrive/TFM/"
if not os.path.exists(save_path):
  os.makedirs(save_path)

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name="Qwen/Qwen2-7B-Instruct"

#Compresión del modelo
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer= AutoTokenizer.from_pretrained(model_name)
model=AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    output_hidden_states=True
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

Cuantización de 4 bits (BitsAndBytesConfig): Los números no se guardan como floats normales sino que se comprimen a 4 bits. Puesto que sólo se pueden representar 16 valores distintos, el número debe ser aproximado.

La cuantización utilizada es NF4 (normal float 4), que está diseñado específicamente para redes neuronales. Los pesos no están distribuidos uniformemente, y NF4 concentra la resolución cerca de 0 para obtener 16 valores posibles con una distribución más parecida a la distribución normal (no uniforme).

Por otro lado, si bien se almacenan los pesos en 4 bits, en los cálculos se usan con 16bits para no perder precisión, se descomprimen temporalmente a float16.

Finalmente, la doble cuantización es útil para cuantizar también los valores de escala que se guardan con los pesos, para continuar reduciendo la memoria sin afectar al rendimiento.

In [ ]:
PROMPTS = {
    'neutral': "",  # Sin instrucción, baseline
    'literary': (
        "You are a literary critic. As you read this text, focus on "
        "narrative structure, character development, and prose style."
    ),
    'cognitive': (
        "You are a cognitive scientist. As you read, focus on the mental "
        "states, beliefs, desires, and intentions of each character."
    ),
    'emotional': (
        "You are reading this story with deep emotional engagement. "
        "Focus on the emotions felt by each character and the emotional "
        "tension of each scene."
    ),
    'syntactic': (
        "You are a linguist analyzing sentence structure. Focus on "
        "grammatical complexity, dependency relations, and word order."
    ),
    'spatial': (
        'Pay close attention to physical movements, spatial locations, '
        'body positions, and the physical environment described in the text.'
    ),
}

In [ ]:
word_texts = np.load('/content/drive/MyDrive/TFM/word_texts.npy', allow_pickle=True)
print(f"'word_texts' loaded with shape: {word_texts.shape}")

'word_texts' loaded with shape: (5176,)


In [ ]:
import torch
import numpy as np
import time

def extract_embeddings(word_texts, system_prompt="", batch_size=1, layer_to_get=-1):

    embeddings = []
    start_time = time.time()

    for i in range(0, len(word_texts), batch_size):

        batch_words = word_texts[i:i+batch_size]

        texts = [
            system_prompt + " " + w
            for w in batch_words
        ]

        tokens = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )

        tokens = {k: v.to(model.device) for k, v in tokens.items()}

        with torch.no_grad():
            output = model(
                **tokens,
                use_cache=False,
                output_hidden_states=True
            )

        # Última capa del transformer
        hidden = output.hidden_states[layer_to_get]

        # Mean pooling con attention mask
        mask = tokens["attention_mask"].unsqueeze(-1)

        batch_emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)

        embeddings.extend(batch_emb.cpu().float().numpy())

        del tokens, output, hidden
        torch.cuda.empty_cache()

        if (i + 1) % 500 == 0:
            print(f"  Palabra {i+1}/{len(word_texts)}")

    print(f"Completado en {(time.time() - start_time)/60:.1f} min")

    return np.array(embeddings)

La función principal genera los embeddings de palabra de la historia, utilizando un modelo Qwen, que consigue la misma eficiencia que modelos más grandes con menos parámetros. En lugar de utilizar la salida de la capa final, se utiliza la de la capa intermedia 14, que según ciertas fuentes de la literatura, como Clark et al. (2019), se centra más en las relaciones semánticas que la última capa, que se encarga de las decisiones finales del modelo.



In [ ]:
for prompt_name, prompt_text in PROMPTS.items():
  filename = f'{save_path}embeddings_{prompt_name}.npy'
  if os.path.exists(filename):
      print(f"Saltando {prompt_name}: Archivo ya existente.")
      continue

  print(f"Procesando: {prompt_name}")
  try:
     embeds = extract_embeddings(word_texts, system_prompt=prompt_text, layer_to_get=-1,  batch_size=1)
     np.save(filename,embeds)
     del embeds
     torch.cuda.empty_cache()
     gc.collect()
  except Exception as e:
        print(f"Error en {prompt_name}: {e}")

Saltando neutral: Archivo ya existente.
Saltando literary: Archivo ya existente.
Saltando cognitive: Archivo ya existente.
Saltando emotional: Archivo ya existente.
Saltando syntactic: Archivo ya existente.
Procesando: spatial
  Palabra 500/5176
  Palabra 1000/5176
  Palabra 1500/5176
  Palabra 2000/5176
  Palabra 2500/5176
  Palabra 3000/5176
  Palabra 3500/5176
  Palabra 4000/5176
  Palabra 4500/5176
  Palabra 5000/5176
Completado en 34.7 min


Esta última celda itera la función para obtener 6 archivos correspondientes a las 6 prompts bajo las cuales el modelo ha generado embeddings. Corresponde a la primera parte del notebook 3; la generación de embeddings, pues el resto del procesamiento no necesita este coste computacional y se puede hacer en local, lo qyue resulta más cómodo teniendo montado el sistema de archivos.